In [1]:
#set up folders, mount drive, imports

import pandas as pd
import numpy as np
import json
from pathlib import Path

# mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# main project folders
ROOT = Path('/content/drive/MyDrive/AI4_Data')
RAW = ROOT / 'Raw'
PROC = ROOT / 'Processed'

# ensure folders exist (safe)
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

# prints to confirm paths
print("ROOT:", ROOT)
print("RAW exists?", RAW.exists())
print("PROC exists?", PROC.exists())

# list current files (helps sanity check)
print("\nFiles in RAW:")
for f in RAW.glob("*"):
    print(" -", f.name)

print("\nFiles in PROCESSED:")
for f in PROC.glob("*"):
    print(" -", f.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT: /content/drive/MyDrive/AI4_Data
RAW exists? True
PROC exists? True

Files in RAW:
 - train_features_0.jsonl
 - train_features_1.jsonl
 - train_features_5.jsonl

Files in PROCESSED:
 - tf0_processed.parquet
 - tf1_ByteEntropy.parquet
 - tf1_processed.parquet


In [2]:
#load the raw JSONL file (line-by-line)
import pandas as pd
import json

rows = []  #will store each JSON line as a Python dictionary

#open train_features_1.jsonl safely
with open(RAW / "train_features_1.jsonl", "r") as f:
    for line in f:
        rows.append(json.loads(line))   #convert each line to dict and store

#convert the list of dictionaries to a DataFrame
tf1 = pd.DataFrame(rows)

#check the shape + first few rows
print("tf1 loaded. Shape:", tf1.shape)
tf1.head()

tf1 loaded. Shape: (158158, 14)


,sha256,md5,appeared,label,avclass,histogram,byteentropy,strings,general,header,section,imports,exports,datadirectories
0,2ef9a92ee6c955364564b0df75ee3753473014b2ba162b...,7e39aeea7bc21d16b8652516a150b282,2018-01,1,sivis,"[60782, 5895, 2020, 1487, 2075, 1367, 1145, 85...","[24434, 11, 18, 5, 9, 8, 35, 21, 3, 4, 8, 4, 2...","{'numstrings': 3863, 'avlength': 17.6435412891...","{'size': 349811, 'vsize': 28672, 'has_debug': ...","{'coff': {'timestamp': 1301832471, 'machine': ...","{'entry': '.code', 'sections': [{'name': '.cod...","{'MSVCRT.dll': ['memset', 'memcpy', '_stricmp'...",[],"[{'name': 'EXPORT_TABLE', 'size': 0, 'virtual_..."
1,50f3f85a10cedf9192f7aa4cd4d2b1ce9e294e23f3dd7e...,09435239e7a0ddfeb78820cf5c31cc06,2018-01,-1,upatre,"[11971, 74, 27, 147, 61, 61, 19, 27, 53, 34, 1...","[2006, 2, 3, 0, 4, 3, 3, 1, 9, 1, 3, 0, 1, 2, ...","{'numstrings': 82, 'avlength': 11.097560975609...","{'size': 33776, 'vsize': 36864, 'has_debug': 0...","{'coff': {'timestamp': 1401281511, 'machine': ...","{'entry': '.text', 'sections': [{'name': '.tex...","{'kernel32.dll': ['Sleep', 'VirtualAlloc', 'Si...",[],"[{'name': 'EXPORT_TABLE', 'size': 0, 'virtual_..."
2,f6c68207b3b395feabcbb029c393607db4ff5227ecd5da...,744cac35cdfa2c3a0672184d433cb93e,2018-01,0,,"[35534, 3516, 1832, 870, 957, 596, 435, 501, 1...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","{'numstrings': 1452, 'avlength': 15.4869146005...","{'size': 161280, 'vsize': 184320, 'has_debug':...","{'coff': {'timestamp': 1512336134, 'machine': ...","{'entry': '.text', 'sections': [{'name': '.tex...","{'WrldFlsh.exe': ['pszStatName', 'dbWeather1',...",[],"[{'name': 'EXPORT_TABLE', 'size': 0, 'virtual_..."
3,3292ee48fac44b7b70bdfde526f71c09e65764d9b582f5...,b6d6f2dc5ef191e23dfc2892ff626168,2018-01,0,,"[3389, 100, 35, 9, 18, 7, 16, 12, 12, 7, 28, 6...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","{'numstrings': 65, 'avlength': 20.030769230769...","{'size': 6144, 'vsize': 32768, 'has_debug': 0,...","{'coff': {'timestamp': 1499089577, 'machine': ...","{'entry': '.text', 'sections': [{'name': '.tex...",{'mscoree.dll': ['_CorExeMain']},[],"[{'name': 'EXPORT_TABLE', 'size': 0, 'virtual_..."
4,b9494c218b74a9774a88307ba83c03da0bee3a80899b47...,b6607d4123b0d3afdbca4bdc9a4bcf7a,2018-01,-1,startsurf,"[256325, 3576, 3209, 3073, 3900, 2695, 2711, 2...","[329804, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","{'numstrings': 5970, 'avlength': 23.8986599664...","{'size': 1242112, 'vsize': 1265664, 'has_debug...","{'coff': {'timestamp': 1515165337, 'machine': ...","{'entry': '.text', 'sections': [{'name': '.tex...","{'KERNEL32.dll': ['CreateEventA', 'CloseHandle...",[],"[{'name': 'EXPORT_TABLE', 'size': 0, 'virtual_..."


In [3]:
#flatten the 'general' column

#convert nested dict -> normal table
gen_tf1 = pd.json_normalize(tf1['general'])

#add prefix so columns become 'gen.*'
gen_tf1 = gen_tf1.add_prefix('gen.')

#quick checks
print("general flattened shape:", gen_tf1.shape)
print("sample columns:", gen_tf1.columns[:10])
gen_tf1.head(3)

general flattened shape: (158158, 10)
sample columns: Index(['gen.size', 'gen.vsize', 'gen.has_debug', 'gen.exports', 'gen.imports',
       'gen.has_relocations', 'gen.has_resources', 'gen.has_signature',
       'gen.has_tls', 'gen.symbols'],
      dtype='object')


,gen.size,gen.vsize,gen.has_debug,gen.exports,gen.imports,gen.has_relocations,gen.has_resources,gen.has_signature,gen.has_tls,gen.symbols
0,349811,28672,0,0,55,0,1,0,0,0
1,33776,36864,0,0,33,0,1,0,0,0
2,161280,184320,0,0,155,1,1,0,0,0


In [4]:
#flatten 'header' column

# convert nested dict -> table
hdr_tf1 = pd.json_normalize(tf1['header'])

# add prefix so columns become 'hdr.*'
hdr_tf1 = hdr_tf1.add_prefix('hdr.')

# check output
print("header flattened shape:", hdr_tf1.shape)
print("sample columns:", hdr_tf1.columns[:10])
hdr_tf1.head(3)

header flattened shape: (158158, 17)
sample columns: Index(['hdr.coff.timestamp', 'hdr.coff.machine', 'hdr.coff.characteristics',
       'hdr.optional.subsystem', 'hdr.optional.dll_characteristics',
       'hdr.optional.magic', 'hdr.optional.major_image_version',
       'hdr.optional.minor_image_version', 'hdr.optional.major_linker_version',
       'hdr.optional.minor_linker_version'],
      dtype='object')


,hdr.coff.timestamp,hdr.coff.machine,hdr.coff.characteristics,hdr.optional.subsystem,hdr.optional.dll_characteristics,hdr.optional.magic,hdr.optional.major_image_version,hdr.optional.minor_image_version,hdr.optional.major_linker_version,hdr.optional.minor_linker_version,hdr.optional.major_operating_system_version,hdr.optional.minor_operating_system_version,hdr.optional.major_subsystem_version,hdr.optional.minor_subsystem_version,hdr.optional.sizeof_code,hdr.optional.sizeof_headers,hdr.optional.sizeof_heap_commit
0,1301832471,I386,"[CHARA_32BIT_MACHINE, RELOCS_STRIPPED, EXECUTA...",WINDOWS_GUI,[],PE32,0,0,2,50,4,0,4,0,8704,1024,4096
1,1401281511,I386,"[CHARA_32BIT_MACHINE, RELOCS_STRIPPED, EXECUTA...",WINDOWS_GUI,[],PE32,4,0,5,12,4,0,4,0,7680,1024,4096
2,1512336134,I386,"[CHARA_32BIT_MACHINE, DLL, EXECUTABLE_IMAGE]",WINDOWS_GUI,[],PE32,0,0,9,0,5,0,5,0,103424,1024,4096


In [5]:
#flatten 'section' column

# section is a nested dictionary -> flatten to table
sect_tf1 = pd.json_normalize(tf1['section'])

# add prefix so cols are 'sect.*'
sect_tf1 = sect_tf1.add_prefix('sect.')

# inspect
print("section flattened shape:", sect_tf1.shape)
print("sample columns:", sect_tf1.columns[:10])
sect_tf1.head(3)

section flattened shape: (158158, 2)
sample columns: Index(['sect.entry', 'sect.sections'], dtype='object')


,sect.entry,sect.sections
0,.code,"[{'name': '.code', 'size': 2048, 'entropy': 5...."
1,.text,"[{'name': '.text', 'size': 7680, 'entropy': 6...."
2,.text,"[{'name': '.text', 'size': 103424, 'entropy': ..."


In [6]:
#flatten 'sect.entry' inside 'section'

# each element in tf1['section'] is a dict that contains 'entry'
# some entries can be missing, so use .get() to avoid errors

entry_list = []

for x in tf1['section']:
    entry_list.append(x.get('entry', {}))  # if no 'entry', return empty dict

# convert list of dicts into DataFrame
sect_entry_tf1 = pd.DataFrame(entry_list)

# add prefix
sect_entry_tf1 = sect_entry_tf1.add_prefix('sect.entry.')

# inspect
print("sect.entry flattened shape:", sect_entry_tf1.shape)
print("sample columns:", sect_entry_tf1.columns[:10])
sect_entry_tf1.head(3)

sect.entry flattened shape: (158158, 1)
sample columns: Index(['sect.entry.0'], dtype='object')


,sect.entry.0
0,.code
1,.text
2,.text


In [7]:
#extract number of sections

#each section dict contains a key 'sections' which is a list
num_sections = [len(x.get('sections', [])) for x in tf1['section']]

#turn into DataFrame
sect_simple_tf1 = pd.DataFrame({
    'sect.num_sections': num_sections
})

#inspect
print("sect.num_sections shape:", sect_simple_tf1.shape)
sect_simple_tf1.head(3)

sect.num_sections shape: (158158, 1)


,sect.num_sections
0,5
1,4
2,5


In [8]:
#count number of exports

#tf1['imports'] is a list OR sometimes missing
num_imports = [len(x) if isinstance(x, list) else 0 for x in tf1['imports']]

imports_simple_tf1 = pd.DataFrame({
    'imports.count': num_imports
})

#inspect
print("imports.count shape:", imports_simple_tf1.shape)
imports_simple_tf1.head(3)

imports.count shape: (158158, 1)


,imports.count
0,0
1,0
2,0


In [9]:
#count number of exports

#same idea as imports: list or missing
num_exports = [len(x) if isinstance(x, list) else 0 for x in tf1['exports']]

exports_simple_tf1 = pd.DataFrame({
    'exports.count': num_exports
})

#inspect
print("exports.count shape:", exports_simple_tf1.shape)
exports_simple_tf1.head(3)

exports.count shape: (158158, 1)


,exports.count
0,0
1,0
2,0


In [10]:
#flatten 'datadirectories' column

#flatten each dict inside the column
dd_tf1 = pd.json_normalize(tf1['datadirectories'])

#add prefix to keep everything organised
dd_tf1 = dd_tf1.add_prefix('dd.')

#inspect
print("datadirectories flattened shape:", dd_tf1.shape)
print("sample columns:", dd_tf1.columns[:10])
dd_tf1.head(3)

datadirectories flattened shape: (158158, 15)
sample columns: Index(['dd.0', 'dd.1', 'dd.2', 'dd.3', 'dd.4', 'dd.5', 'dd.6', 'dd.7', 'dd.8',
       'dd.9'],
      dtype='object')


,dd.0,dd.1,dd.2,dd.3,dd.4,dd.5,dd.6,dd.7,dd.8,dd.9,dd.10,dd.11,dd.12,dd.13,dd.14
0,"{'name': 'EXPORT_TABLE', 'size': 0, 'virtual_a...","{'name': 'IMPORT_TABLE', 'size': 140, 'virtual...","{'name': 'RESOURCE_TABLE', 'size': 700, 'virtu...","{'name': 'EXCEPTION_TABLE', 'size': 0, 'virtua...","{'name': 'CERTIFICATE_TABLE', 'size': 0, 'virt...","{'name': 'BASE_RELOCATION_TABLE', 'size': 0, '...","{'name': 'DEBUG', 'size': 0, 'virtual_address'...","{'name': 'ARCHITECTURE', 'size': 0, 'virtual_a...","{'name': 'GLOBAL_PTR', 'size': 0, 'virtual_add...","{'name': 'TLS_TABLE', 'size': 0, 'virtual_addr...","{'name': 'LOAD_CONFIG_TABLE', 'size': 0, 'virt...","{'name': 'BOUND_IMPORT', 'size': 0, 'virtual_a...","{'name': 'IAT', 'size': 244, 'virtual_address'...","{'name': 'DELAY_IMPORT_DESCRIPTOR', 'size': 0,...","{'name': 'CLR_RUNTIME_HEADER', 'size': 0, 'vir..."
1,"{'name': 'EXPORT_TABLE', 'size': 0, 'virtual_a...","{'name': 'IMPORT_TABLE', 'size': 80, 'virtual_...","{'name': 'RESOURCE_TABLE', 'size': 14776, 'vir...","{'name': 'EXCEPTION_TABLE', 'size': 0, 'virtua...","{'name': 'CERTIFICATE_TABLE', 'size': 0, 'virt...","{'name': 'BASE_RELOCATION_TABLE', 'size': 0, '...","{'name': 'DEBUG', 'size': 0, 'virtual_address'...","{'name': 'ARCHITECTURE', 'size': 0, 'virtual_a...","{'name': 'GLOBAL_PTR', 'size': 0, 'virtual_add...","{'name': 'TLS_TABLE', 'size': 0, 'virtual_addr...","{'name': 'LOAD_CONFIG_TABLE', 'size': 0, 'virt...","{'name': 'BOUND_IMPORT', 'size': 0, 'virtual_a...","{'name': 'IAT', 'size': 144, 'virtual_address'...","{'name': 'DELAY_IMPORT_DESCRIPTOR', 'size': 0,...","{'name': 'CLR_RUNTIME_HEADER', 'size': 0, 'vir..."
2,"{'name': 'EXPORT_TABLE', 'size': 0, 'virtual_a...","{'name': 'IMPORT_TABLE', 'size': 180, 'virtual...","{'name': 'RESOURCE_TABLE', 'size': 2264, 'virt...","{'name': 'EXCEPTION_TABLE', 'size': 0, 'virtua...","{'name': 'CERTIFICATE_TABLE', 'size': 0, 'virt...","{'name': 'BASE_RELOCATION_TABLE', 'size': 6016...","{'name': 'DEBUG', 'size': 0, 'virtual_address'...","{'name': 'ARCHITECTURE', 'size': 0, 'virtual_a...","{'name': 'GLOBAL_PTR', 'size': 0, 'virtual_add...","{'name': 'TLS_TABLE', 'size': 0, 'virtual_addr...","{'name': 'LOAD_CONFIG_TABLE', 'size': 64, 'vir...","{'name': 'BOUND_IMPORT', 'size': 0, 'virtual_a...","{'name': 'IAT', 'size': 652, 'virtual_address'...","{'name': 'DELAY_IMPORT_DESCRIPTOR', 'size': 0,...","{'name': 'CLR_RUNTIME_HEADER', 'size': 0, 'vir..."


In [11]:
#flatten 'histogram' column

#flatten dict of histogram values
hist_tf1 = pd.json_normalize(tf1['histogram'])

#add prefix
hist_tf1 = hist_tf1.add_prefix('hist.')

#inspect
print("histogram flattened shape:", hist_tf1.shape)
print("sample columns:", hist_tf1.columns[:10])
hist_tf1.head(3)

histogram flattened shape: (158158, 256)
sample columns: Index(['hist.0', 'hist.1', 'hist.2', 'hist.3', 'hist.4', 'hist.5', 'hist.6',
       'hist.7', 'hist.8', 'hist.9'],
      dtype='object')


,hist.0,hist.1,hist.2,hist.3,hist.4,hist.5,hist.6,hist.7,hist.8,hist.9,...,hist.246,hist.247,hist.248,hist.249,hist.250,hist.251,hist.252,hist.253,hist.254,hist.255
0,{},{},{},{},{},{},{},{},{},{},...,{},{},{},{},{},{},{},{},{},{}
1,{},{},{},{},{},{},{},{},{},{},...,{},{},{},{},{},{},{},{},{},{}
2,{},{},{},{},{},{},{},{},{},{},...,{},{},{},{},{},{},{},{},{},{}


In [12]:
#load Byte Entropy features for TF1 (processed seperately)

#parquet already exists
byte_tf1 = pd.read_parquet(PROC / "tf1_ByteEntropy.parquet")

#inspect
print("byte entropy shape:", byte_tf1.shape)
print("sample columns:", byte_tf1.columns[:10])
byte_tf1.head(3)

byte entropy shape: (158158, 256)
sample columns: Index(['byte.0', 'byte.1', 'byte.2', 'byte.3', 'byte.4', 'byte.5', 'byte.6',
       'byte.7', 'byte.8', 'byte.9'],
      dtype='object')


,byte.0,byte.1,byte.2,byte.3,byte.4,byte.5,byte.6,byte.7,byte.8,byte.9,...,byte.246,byte.247,byte.248,byte.249,byte.250,byte.251,byte.252,byte.253,byte.254,byte.255
0,24434.0,11.0,18.0,5.0,9.0,8.0,35.0,21.0,3.0,4.0,...,16462.0,15966.0,14060.0,13914.0,13731.0,13957.0,14147.0,13756.0,13708.0,13644.0
1,2006.0,2.0,3.0,0.0,4.0,3.0,3.0,1.0,9.0,1.0,...,371.0,332.0,236.0,212.0,307.0,319.0,347.0,173.0,573.0,330.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
#flatten 'strings' column

#'strings' is a dictionary, so flatten it like the others
str_tf1 = pd.json_normalize(tf1['strings']).add_prefix('str.')

#inspect
print("strings flattened shape:", str_tf1.shape)
print("sample columns:", str_tf1.columns[:10])
str_tf1.head(3)

strings flattened shape: (158158, 9)
sample columns: Index(['str.numstrings', 'str.avlength', 'str.printabledist', 'str.printables',
       'str.entropy', 'str.paths', 'str.urls', 'str.registry', 'str.MZ'],
      dtype='object')


,str.numstrings,str.avlength,str.printabledist,str.printables,str.entropy,str.paths,str.urls,str.registry,str.MZ
0,3863,17.643541,"[7399, 89, 373, 105, 116, 114, 101, 94, 430, 4...",68157,5.683164,6,22,0,7
1,82,11.097561,"[23, 3, 0, 2, 2, 0, 1, 1, 0, 0, 3, 3, 2, 5, 12...",910,5.454550,0,0,0,2
2,1452,15.486915,"[844, 28, 94, 26, 36, 51, 77, 27, 37, 31, 14, ...",22487,5.668189,0,204,0,2


In [14]:
#combine all flattened columns

#merge everything column-wise (axis=1)
final_tf1 = pd.concat([
    tf1[['label']],         #keep label first

    gen_tf1,                #flattened general
    hdr_tf1,                #flattened header
    sect_tf1,               #flattened section
    sect_entry_tf1,         #flattened sect.entry
    sect_simple_tf1,        #number of sections
    imports_simple_tf1,     #imports.count
    exports_simple_tf1,     #exports.count
    dd_tf1,                 #datadirectories
    hist_tf1,               #histogram
    byte_tf1,               #byte entropy
    str_tf1                 #strings
], axis=1)

#check final shape and structure
print("Final TF1 merged shape:", final_tf1.shape)
final_tf1.head(3)

Final TF1 merged shape: (158158, 570)


,label,gen.size,gen.vsize,gen.has_debug,gen.exports,gen.imports,gen.has_relocations,gen.has_resources,gen.has_signature,gen.has_tls,...,byte.255,str.numstrings,str.avlength,str.printabledist,str.printables,str.entropy,str.paths,str.urls,str.registry,str.MZ
0,1,349811,28672,0,0,55,0,1,0,0,...,13644.0,3863,17.643541,"[7399, 89, 373, 105, 116, 114, 101, 94, 430, 4...",68157,5.683164,6,22,0,7
1,-1,33776,36864,0,0,33,0,1,0,0,...,330.0,82,11.097561,"[23, 3, 0, 2, 2, 0, 1, 1, 0, 0, 3, 3, 2, 5, 12...",910,5.454550,0,0,0,2
2,0,161280,184320,0,0,155,1,1,0,0,...,0.0,1452,15.486915,"[844, 28, 94, 26, 36, 51, 77, 27, 37, 31, 14, ...",22487,5.668189,0,204,0,2


In [15]:
#identify columns whose dtype is 'object' AND contain nested/struct-like values
problem_cols = []

for col in final_tf1.columns:
    if final_tf1[col].dtype == 'object':
        #check first non-null value
        val = final_tf1[col].dropna().iloc[0] if final_tf1[col].dropna().shape[0] > 0 else None
        if isinstance(val, dict):
            problem_cols.append(col)

problem_cols

['dd.0',
 'dd.1',
 'dd.2',
 'dd.3',
 'dd.4',
 'dd.5',
 'dd.6',
 'dd.7',
 'dd.8',
 'dd.9',
 'dd.10',
 'dd.11',
 'dd.12',
 'dd.13',
 'dd.14',
 'hist.0',
 'hist.1',
 'hist.2',
 'hist.3',
 'hist.4',
 'hist.5',
 'hist.6',
 'hist.7',
 'hist.8',
 'hist.9',
 'hist.10',
 'hist.11',
 'hist.12',
 'hist.13',
 'hist.14',
 'hist.15',
 'hist.16',
 'hist.17',
 'hist.18',
 'hist.19',
 'hist.20',
 'hist.21',
 'hist.22',
 'hist.23',
 'hist.24',
 'hist.25',
 'hist.26',
 'hist.27',
 'hist.28',
 'hist.29',
 'hist.30',
 'hist.31',
 'hist.32',
 'hist.33',
 'hist.34',
 'hist.35',
 'hist.36',
 'hist.37',
 'hist.38',
 'hist.39',
 'hist.40',
 'hist.41',
 'hist.42',
 'hist.43',
 'hist.44',
 'hist.45',
 'hist.46',
 'hist.47',
 'hist.48',
 'hist.49',
 'hist.50',
 'hist.51',
 'hist.52',
 'hist.53',
 'hist.54',
 'hist.55',
 'hist.56',
 'hist.57',
 'hist.58',
 'hist.59',
 'hist.60',
 'hist.61',
 'hist.62',
 'hist.63',
 'hist.64',
 'hist.65',
 'hist.66',
 'hist.67',
 'hist.68',
 'hist.69',
 'hist.70',
 'hist.71',
 'hist

In [16]:
#fix: convert problematic struct columns to numeric

cols_to_fix = problem_cols  #from step A output

for col in cols_to_fix:
    #if value is dict or weird object, replace with 0
    final_tf1[col] = final_tf1[col].apply(
        lambda x: 0 if isinstance(x, dict) else x
    )

    #then force numeric (non-convertible → NaN → filled with 0)
    final_tf1[col] = pd.to_numeric(final_tf1[col], errors='coerce').fillna(0)

In [17]:
#standardise all column types to float32

#keep label as int
final_tf1['label'] = final_tf1['label'].astype('int8')

#everything else → float32
for col in final_tf1.columns:
    if col != 'label':
        final_tf1[col] = pd.to_numeric(final_tf1[col], errors='coerce').astype('float32')

In [18]:
output_path = PROC / "tf1_processed.parquet"
chunk_size = 20000
num_rows = final_tf1.shape[0]

import pyarrow as pa
import pyarrow.parquet as pq

writer = None

for start in range(0, num_rows, chunk_size):
    end = min(start + chunk_size, num_rows)
    chunk = final_tf1.iloc[start:end]
    table = pa.Table.from_pandas(chunk, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(output_path, table.schema)

    writer.write_table(table)

if writer is not None:
    writer.close()

print("SAVED SUCCESSFULLY:", output_path)

SAVED SUCCESSFULLY: /content/drive/MyDrive/AI4_Data/Processed/tf1_processed.parquet


In [20]:
import pyarrow.parquet as pq

path = "/content/drive/MyDrive/AI4_Data/Processed/tf1_processed.parquet"  # adjust if needed
pf = pq.ParquetFile(path)

# rows
print("Samples (rows):", pf.metadata.num_rows)

# columns
schema = pf.schema_arrow
total_cols = len(schema)
print("Total columns:", total_cols)

# features (assuming 1 label column)
print("Features (assuming 1 label):", total_cols - 1)

print("First 30 column names:")
print(schema.names[:30])


Samples (rows): 158158
Total columns: 570
Features (assuming 1 label): 569
First 30 column names:
['label', 'gen.size', 'gen.vsize', 'gen.has_debug', 'gen.exports', 'gen.imports', 'gen.has_relocations', 'gen.has_resources', 'gen.has_signature', 'gen.has_tls', 'gen.symbols', 'hdr.coff.timestamp', 'hdr.coff.machine', 'hdr.coff.characteristics', 'hdr.optional.subsystem', 'hdr.optional.dll_characteristics', 'hdr.optional.magic', 'hdr.optional.major_image_version', 'hdr.optional.minor_image_version', 'hdr.optional.major_linker_version', 'hdr.optional.minor_linker_version', 'hdr.optional.major_operating_system_version', 'hdr.optional.minor_operating_system_version', 'hdr.optional.major_subsystem_version', 'hdr.optional.minor_subsystem_version', 'hdr.optional.sizeof_code', 'hdr.optional.sizeof_headers', 'hdr.optional.sizeof_heap_commit', 'sect.entry', 'sect.sections']


In [21]:
import pyarrow.parquet as pq
import pyarrow.compute as pc

path = "/content/drive/MyDrive/AI4_Data/Processed/tf1_processed.parquet"
table = pq.read_table(path, columns=["label"])
print(pc.value_counts(table["label"]))

-- is_valid: all not null
-- child 0 type: int8
  [
    1,
    -1,
    0
  ]
-- child 1 type: int64
  [
    63713,
    42107,
    52338
  ]
